# L1c: A First Tested Engineering Calculation

We will translate the ideal-gas relation, its unit contract, and its admissible input domain into a small Julia function with executable tests.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __State the contract before writing code:__ Fix the units, the admissible input domain, and the return value of an engineering calculation before translating it into a Julia function. The contract is what makes the function testable.
> * __Reject inputs the model cannot support:__ Raise a clear error when an argument violates a physical assumption, rather than returning a number that looks plausible and means nothing.
> * __Test known behavior, not just execution:__ Check a reference case with an appropriate numerical tolerance and interpret the result in engineering units, so that a passing test says something about the calculation.

Let's get started!
___

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Besides Julia's `Base` library, this lab uses [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) for the executable checks at the end. `Include.jl` also pulls in [`src/Compute.jl`](src/Compute.jl), which is where the calculation we are about to build actually lives.

___

## Contract before code

An engineering calculation is not just a formula. It is a formula plus a statement of what each symbol means, which values are admissible, and what comes back out. Writing that down first is not bureaucracy: it is what turns a formula into something a test can check.

The [ideal gas law](https://en.wikipedia.org/wiki/Ideal_gas_law) relates the state variables of a gas:

$$
PV = nRT
$$

We want pressure, so rearrange for $P$:

$$
P = \frac{nRT}{V}
$$

That is the whole calculation. The contract is the part that takes thought.

> __The contract:__
>
> * $n$ — amount of substance, in mol. Finite and strictly positive.
> * $T$ — absolute temperature, in K. Finite and strictly positive; at or below absolute zero the model does not apply.
> * $V$ — volume, in $\mathrm{m^{3}}$. Finite and strictly positive; a gas with no volume has no pressure to report.
> * $R$ — the [universal gas constant](https://en.wikipedia.org/wiki/Gas_constant), $8.31446261815324\;\mathrm{Pa\,m^{3}\,mol^{-1}\,K^{-1}}$. Exact by definition since the 2019 SI redefinition, so it is a default rather than an argument you must supply.
> * Returns $P$ — pressure, in Pa.

Each clause above becomes either a line of validation code or a test. Watch for that correspondence as we go: it is the point of the lab.

___

## Implement the function

The function lives in [`src/Compute.jl`](src/Compute.jl) rather than in a notebook cell. That is deliberate. Code in a source file can be imported by other notebooks, checked by the validation suite, and edited with real tooling; code in a cell exists only as long as the kernel does.

Right now that file holds a signature, a docstring, and two `TODO` comments. Filling them in is the work of this lab.

> __What to write:__
>
> * __Validate the inputs.__ For each of `amount_mol`, `temperature_K`, `volume_m3` and `gas_constant`, throw an [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError) naming that argument when the value is not finite, or not strictly positive. [The `isfinite(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.isfinite) does the first check. Writing the four checks as one loop over a `NamedTuple` is shorter than writing them out four times.
> * __Return the pressure.__ Compute $P = nRT/V$ and return it as a `Float64`.

Open the file in your editor, make both changes, then restart the kernel and run this notebook from the top. The test cell further down is your specification: when it goes green, you are done.

Until then the notebook stops at the next cell with a "not implemented yet" error, which is the expected starting state. Let's read what you are starting from:

In [ ]:
implementation_path = joinpath(CHEME5800_L1C_ROOT, "src", "Compute.jl")
implementation_preview = read(implementation_path, String)
println(implementation_preview) # println, not the bare value: we want readable lines, not one escaped string

___

## Compute a reference case

A new function deserves a case whose answer you already know. One mole of an ideal gas at $273.15\;\mathrm{K}$ occupies about $22.414\;\mathrm{L}$ at one atmosphere. That is the molar volume at standard temperature and pressure, and it is worth carrying in your head.

> __Where does $22.414\;\mathrm{L}$ come from?__
>
> It is not a measurement, it is a rearrangement: $V = nRT/P$ with $n = 1\;\mathrm{mol}$, $T = 273.15\;\mathrm{K}$, and $P = 101325\;\mathrm{Pa}$. Feeding that volume back into our function should therefore return one atmosphere. This confirms that the arithmetic inverts correctly; it is not independent evidence about the behaviour of any real gas.

We store the result in the `pressure_Pa::Float64` variable, and its kilopascal equivalent in `pressure_kPa::Float64`:

In [ ]:
amount_mol = 1.0
temperature_K = 273.15
volume_m3 = 0.02241396954
pressure_Pa = ideal_gas_pressure(amount_mol, temperature_K, volume_m3)
pressure_kPa = pressure_Pa / 1000
(pressure_Pa = pressure_Pa, pressure_kPa = pressure_kPa)

___

## Test the contract

Now we turn each clause of the contract into a check. The first test pins the reference value, the second fixes the return type, and the last three confirm that inadmissible input raises rather than quietly returning a number.

> __Why `isapprox` and not `==`?__
>
> Floating-point arithmetic rounds at every step, so two calculations that agree exactly on paper can differ in their final bits. [The `isapprox(...)` function](https://docs.julialang.org/en/v1/base/math/#Base.isapprox) compares within a tolerance instead of demanding identical bits; `rtol = 1e-8` here asks for agreement to eight significant figures. We will meet the one case where `==` is safe a few cells from now.

[The `@test_throws` macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test_throws) inverts the usual question: it passes only when the expression raises the [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError) we specified. A function that accepts a negative temperature fails this suite.

Do all five clauses hold? Until you complete both `TODO`s these tests fail, and the failure messages tell you which clause is still unmet.

In [ ]:

@testset "ideal-gas pressure contract" begin
    @test isapprox(pressure_Pa, 101_325.0; rtol = 1e-8)
    @test ideal_gas_pressure(2, 300, 0.05) isa Float64
    @test_throws ArgumentError ideal_gas_pressure(0, 300, 0.05)
    @test_throws ArgumentError ideal_gas_pressure(1, -10, 0.05)
    @test_throws ArgumentError ideal_gas_pressure(1, 300, Inf)
end

___

## Change, predict, compute, explain

The reference case shows the function is right at one point. A more useful question is whether it responds correctly when something changes.

Before running the next cell, commit to a prediction: amount and volume stay fixed, temperature doubles from $300\;\mathrm{K}$ to $600\;\mathrm{K}$. What happens to the pressure, and why?

In [ ]:
baseline_pressure = ideal_gas_pressure(1.0, 300.0, 0.025)
hot_pressure = ideal_gas_pressure(1.0, 600.0, 0.025)
pressure_ratio = hot_pressure / baseline_pressure
@test pressure_ratio == 2.0
pressure_ratio

> __Wait — didn't we just say not to use `==`?__ We did, and this is the exception worth understanding. Doubling is _exact_ in binary floating point: $600.0 = 2\times 300.0$ with no rounding, and scaling by a power of two only increments the exponent field, leaving every significand bit untouched. So the two pressures have identical significands and the ratio is exactly `2.0`.
>
> This is fragile in a way that is easy to miss. Change `600.0` to `900.0` and predict the result before you run it: the ratio comes back `2.9999999999999996`, and `== 3.0` fails. Tripling is not a power of two, so the rounding no longer cancels. Exact equality is safe only when you can point to the reason — otherwise, reach for `isapprox`.

___

## Interpretation

One mole in 22.4 L at 273.15 K came back as roughly $101$ kPa — about one atmosphere, which is the number to carry in your head as a sanity check for gas-phase work. The implementation reproduces that reference pressure and the expected proportional response to temperature. It does **not** establish that a real gas is ideal under every condition. The model assumption remains part of the result.

___

## Summary
Translating an equation into code is the easy half; stating what the code promises, and testing that promise, is the half that makes the result usable.

> __Key Takeaways:__
>
> * **Units and domain come first:** Deciding what the arguments mean and which values are admissible turns an equation into a function whose behavior can be checked.
> * **Invalid input deserves an error:** Rejecting a non-physical argument with an `ArgumentError` is more useful than returning a finite number that carries no meaning.
> * **A passing test validates the implementation, not the model:** These tests show the function matches its contract. They say nothing about whether a real gas behaves ideally under the conditions you care about.

Every calculation in this course carries assumptions. Writing them down as a contract is what lets a test tell you when they have been violated.
___